# Reverse-QTY series C MD, part 3: the kill criterion

Taner Karagol and Alper Karagol.

**Run all.** No GPU needed and nothing to fill in unless you want to move the
thresholds. It reads the two reports part 2 wrote, applies the criterion stated
before anything was built, writes `md/results/kill_criterion.json` and draws
`figures/kill_criterion.png` and `.svg`.

## The criterion

The converted dimer has to hold its interface in water over 100 ns, measured against
the native dimer run in the same conditions. The thresholds are in the settings cell
and are declared before the runs are read, as the paper-idea screen requires:

- HOLDS, when the converted dimer keeps at least the hold fraction of the native
  buried interfacial area and of the native inter-subunit contact count, both over
  the scored window and over its last tenth.
- STOP, when either falls below the stop fraction, or when the subunits are no
  longer in contact at the end of the run.
- INCONCLUSIVE between the two, which asks for longer sampling or replicates instead
  of a verdict.

The last tenth is scored separately so that an interface which is still coming apart
at the end does not pass on its average.

- CONTINUE, when the runs are shorter than `MIN_NS_FOR_HOLD` and nothing has fired
  the stop arm. The criterion is one-sided below that length: a short run can stop
  the series, because an interface that is coming apart shows it early, and it cannot
  pass it, because not falling apart in a few tens of nanoseconds is not evidence of
  holding and the published companion study ran 100 ns. An early look that can only
  stop never makes a pass easier to get.

## What it does not decide

The polar pairs of the native interface have no donor in the converted build,
because deleting that hydroxyl and that amide is the substitution itself. Their loss
is reported and is not evidence either way, so it carries no weight in the verdict.
The per-position distances to the partner subunit are reported for all three
converted interface positions, since those exist in both builds.

A STOP here stops the series on its own terms. What remains is the restricted
conversion, the same rule with the interface positions excluded, written as a stated
departure from the published rule and not as a null.


In [ ]:
#@title 1. Settings and the declared thresholds
PROJECT_DIR     = "" #@param {type:"string"}
#@markdown The project folder, used when it is not where DRIVE_ROOT and
#@markdown PROJECT_FOLDER below point. Set it to wherever you put this folder,
#@markdown on Drive or on the Colab disk, and every other path follows from it.
#@markdown Left empty it falls back to those two.
DRIVE_ROOT      = "/content/drive/MyDrive"  #@param {type:"string"}
PROJECT_FOLDER  = "14-3-3-rQTY" #@param {type:"string"}
HOLD_FRACTION   = 0.50   #@param {type:"number"}
#@markdown The converted dimer passes when it keeps at least this fraction of the
#@markdown native buried interfacial area and of the native inter-subunit contact
#@markdown count. Declared here before the runs are read, as the screen requires.
FAIL_FRACTION   = 0.25   #@param {type:"number"}
#@markdown Below this fraction the interface is gone and the series stops. Between
#@markdown the two the result is inconclusive and needs longer sampling or replicates
#@markdown rather than a verdict.
SEPARATED_A     = 5.0    #@param {type:"number"}
#@markdown A shortest inter-subunit distance above this in the scored window means
#@markdown the subunits are no longer in contact at all.
TAIL_FRACTION   = 0.10   #@param {type:"number"}
#@markdown The last tenth of each trajectory is scored on its own, so an interface
#@markdown that is still coming apart at the end does not pass on its average.
MIN_NS_FOR_HOLD = 100.0  #@param {type:"number"}
#@markdown The criterion is one-sided below this length. A short run can stop the
#@markdown series, since an interface that is coming apart shows it early, and it
#@markdown cannot pass it, since not falling apart in a few tens of nanoseconds is
#@markdown not evidence of holding and the published companion study ran 100 ns. An
#@markdown early look that can only stop never makes a pass easier to get.

import os
PROJECT = PROJECT_DIR.strip() or os.path.join(DRIVE_ROOT, PROJECT_FOLDER)
MD = os.path.join(PROJECT, "md")
RESULTS = os.path.join(MD, "results")
FIGURES = os.path.join(PROJECT, "figures")
BUILD_ORDER = ["native", "converted"]
print("project    ", PROJECT)
print("thresholds  hold at or above %.2f of native, stop below %.2f, separated above "
      "%.1f A" % (HOLD_FRACTION, FAIL_FRACTION, SEPARATED_A))


In [ ]:
#@title 2. Read both runs and apply the criterion
from google.colab import drive
drive.mount('/content/drive')

import collections, datetime, glob, json, os, sys
import numpy as np

os.makedirs(FIGURES, exist_ok=True)


def fatal(msg):
    raise SystemExit("STOP: " + msg)


REP = {}
for b in BUILD_ORDER:
    p = os.path.join(RESULTS, "%s_report.json" % b)
    if not os.path.isfile(p):
        fatal("no %s. Run the part 2 notebook for the %s build first; this one "
              "compares the two and cannot do it from one." % (p, b))
    REP[b] = json.load(open(p))

for b in BUILD_ORDER:
    if REP[b]["frames"] < 100:
        fatal("the %s run holds %d frames, which is too few to score"
              % (b, REP[b]["frames"]))
ns = {b: REP[b]["ns"] for b in BUILD_ORDER}
if abs(ns["native"] - ns["converted"]) > 0.05 * max(ns.values()):
    fatal("the two runs span %s ns, and the comparison needs matched sampling. Let "
          "the shorter one catch up before scoring." % ns)
SHORT = min(ns.values()) < MIN_NS_FOR_HOLD
if REP["native"]["converted_interface_positions"] != \
        REP["converted"]["converted_interface_positions"]:
    fatal("the two runs disagree about which converted positions are in the "
          "interface, so they are not the same system")
POS = REP["native"]["converted_interface_positions"]
print("both runs span %.0f ns, %d and %d frames, interface positions %s"
      % (ns["native"], REP["native"]["frames"], REP["converted"]["frames"], POS))


def series(build, name, column=1, scale=1.0):
    base = REP[build].get("run")
    if not base:
        fatal("the %s report does not say which folder it ran from, so it was "
              "written by an earlier part 2. Re-run part 2 for that build." % build)
    p = os.path.join(base, name)
    if not os.path.isfile(p):
        fatal("no %s. The part 2 run stages every xvg it writes to Drive, so a "
              "missing one means that staging did not finish." % p)
    D = np.loadtxt(p, comments=["#", "@"])
    if D.ndim == 1:
        D = D.reshape(1, -1)
    return D[:, 0], D[:, column] * scale


def buried_series(build):
    """Area buried per subunit, the convention code/iface_1433.py uses."""
    t, ab = series(build, "sasa_dimer.xvg")
    _, a = series(build, "sasa_sub1.xvg")
    _, b = series(build, "sasa_sub2.xvg")
    n = min(len(ab), len(a), len(b))
    return t[:n], (a[:n] + b[:n] - ab[:n]) * 100.0 / 2.0


def scored(v, burn, tail=False):
    v = np.asarray(v, dtype=float)
    k = int(len(v) * burn)
    w = v[k:]
    if tail:
        w = w[-max(1, int(len(w) * TAIL_FRACTION)):]
    return w


BURN = REP["native"]["burn"] / float(REP["native"]["frames"])
S = {}
for b in BUILD_ORDER:
    t, bur = buried_series(b)
    _, nc = series(b, "numcont_dimer.xvg")
    _, md = series(b, "mindist_dimer.xvg", scale=10.0)
    S[b] = dict(t=t, buried=bur, contacts=nc, mindist=md)
    print("  %-10s buried %6.0f A2 per subunit over the scored window, %6.0f over "
          "its last tenth"
          % (b, scored(bur, BURN).mean(), scored(bur, BURN, tail=True).mean()))

ratio = {}
for key in ("buried", "contacts"):
    nat = scored(S["native"][key], BURN).mean()
    con = scored(S["converted"][key], BURN).mean()
    nat_tail = scored(S["native"][key], BURN, tail=True).mean()
    con_tail = scored(S["converted"][key], BURN, tail=True).mean()
    if nat <= 0:
        fatal("the native run gives %s of %g, so there is nothing to compare against"
              % (key, nat))
    ratio[key] = dict(native=float(nat), converted=float(con),
                      fraction=float(con / nat),
                      native_tail=float(nat_tail), converted_tail=float(con_tail),
                      fraction_tail=float(con_tail / nat_tail) if nat_tail > 0 else 0.0)
    print("  %-9s converted keeps %.2f of native over the window and %.2f over the "
          "last tenth" % (key, ratio[key]["fraction"], ratio[key]["fraction_tail"]))

sep = float(scored(S["converted"]["mindist"], BURN, tail=True).mean())
worst = min(ratio[k]["fraction"] for k in ratio)
worst_tail = min(ratio[k]["fraction_tail"] for k in ratio)
if sep > SEPARATED_A:
    verdict, why = "STOP", ("the converted subunits are %.1f A apart over the last "
                            "tenth of the run, so they are no longer in contact" % sep)
elif worst < FAIL_FRACTION or worst_tail < FAIL_FRACTION:
    verdict, why = "STOP", ("the converted dimer keeps %.2f of the native interface "
                            "over the window and %.2f over its last tenth, below the "
                            "declared %.2f" % (worst, worst_tail, FAIL_FRACTION))
elif SHORT:
    verdict, why = "CONTINUE", ("the runs span %.0f ns and nothing has fired the stop "
                                "arm. The converted dimer keeps %.2f of the native "
                                "interface so far, and a pass is withheld until %.0f "
                                "ns, so carry both runs on."
                                % (min(ns.values()), worst, MIN_NS_FOR_HOLD))
elif worst >= HOLD_FRACTION and worst_tail >= HOLD_FRACTION:
    verdict, why = "HOLDS", ("the converted dimer keeps %.2f of the native interface "
                             "over the window and %.2f over its last tenth, at or "
                             "above the declared %.2f"
                             % (worst, worst_tail, HOLD_FRACTION))
else:
    verdict, why = "INCONCLUSIVE", ("the converted dimer keeps %.2f of the native "
                                    "interface over the window and %.2f over its last "
                                    "tenth, between the declared %.2f and %.2f, so "
                                    "this needs longer sampling or replicates instead "
                                    "of a verdict"
                                    % (worst, worst_tail, FAIL_FRACTION, HOLD_FRACTION))

per_pos = {}
for k in sorted(REP["native"]["per_position_mindist_A"]):
    n = REP["native"]["per_position_mindist_A"][k]["mean"]
    c = REP["converted"]["per_position_mindist_A"][k]["mean"]
    per_pos[k] = dict(native_A=n, converted_A=c, change_A=c - n)

out = dict(verdict=verdict, reason=why,
           thresholds=dict(hold=HOLD_FRACTION, stop=FAIL_FRACTION,
                           separated_A=SEPARATED_A, tail_fraction=TAIL_FRACTION,
                           burn_fraction=BURN, min_ns_for_hold=MIN_NS_FOR_HOLD),
           one_sided=bool(SHORT),
           ns=ns, frames={b: REP[b]["frames"] for b in BUILD_ORDER},
           interface=ratio, converted_mindist_tail_A=sep,
           per_position_mindist=per_pos,
           rmsd_A={b: REP[b]["rmsd_A"] for b in BUILD_ORDER},
           native_polar_pairs=REP["native"]["polar_pairs"],
           converted_polar_pairs=REP["converted"]["polar_pairs"],
           written=datetime.datetime.now().isoformat(timespec="seconds"))
json.dump(out, open(os.path.join(RESULTS, "kill_criterion.json"), "w"), indent=2)

print("\n" + "=" * 72)
print("KILL CRITERION: %s" % verdict)
print("=" * 72)
print(why)
print("\nbackbone RMSD: native %.2f +/- %.2f A, converted %.2f +/- %.2f A"
      % (REP["native"]["rmsd_A"]["mean"], REP["native"]["rmsd_A"]["sd"],
         REP["converted"]["rmsd_A"]["mean"], REP["converted"]["rmsd_A"]["sd"]))
print("\nshortest distance from each converted interface position to the partner "
      "subunit, angstrom:")
for k, v in per_pos.items():
    print("   %-10s native %5.2f   converted %5.2f   change %+5.2f"
          % (k, v["native_A"], v["converted_A"], v["change_A"]))
if REP["native"]["polar_pairs"]:
    print("\nthe polar pairs of the native interface, and what the conversion does "
          "to them:")
    for pretty, s in sorted(REP["native"]["polar_pairs"].items()):
        gone = pretty not in REP["converted"]["polar_pairs"]
        print("   %-34s %.2f A, within 3.5 A in %.0f per cent of frames%s"
              % (pretty, s["mean"], 100.0 * s["occupancy_3_5"],
                 ", donor deleted by the substitution" if gone else ""))
print("\nwrote %s" % os.path.join(RESULTS, "kill_criterion.json"))
if verdict == "STOP":
    print("\nThe series stops here on its own criterion. What remains is the "
          "restricted conversion, the same rule with the interface positions "
          "excluded, written as a departure from the published rule.")
elif verdict == "CONTINUE":
    print("\nRaise NS_PER_BUILD in both part 2 notebooks to %.0f and press Run all "
          "again. The finished chunks are skipped, so they carry on from where they "
          "stopped." % MIN_NS_FOR_HOLD)


In [ ]:
#@title 3. The figure
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

COL = {"native": "#4C6EF5", "converted": "#E8590C"}
fig, ax = plt.subplots(1, 3, figsize=(13.5, 4.0), facecolor="white")
panels = (("buried", "Buried interfacial area per subunit, $\\mathrm{\\AA}^2$"),
          ("contacts", "Inter-subunit contacts within 5 $\\mathrm{\\AA}$"),
          ("mindist", "Shortest inter-subunit distance, $\\mathrm{\\AA}$"))
for a, (key, label) in zip(ax, panels):
    ends = []
    for b in BUILD_ORDER:
        t = S[b]["t"] / 1000.0
        v = S[b][key]
        n = min(len(t), len(v))
        a.plot(t[:n], v[:n], lw=0.8, color=COL[b], label=b)
        ends.append((b, float(t[n - 1]), float(v[n - 1])))
    ends.sort(key=lambda e: e[2])
    span = max(e[2] for e in ends) - min(e[2] for e in ends)
    for i, (b, x, y) in enumerate(ends):
        dy = 0 if span > 0.02 * (abs(y) + 1e-9) else (-7 + 14 * i)
        a.annotate(b, xy=(x, y), xytext=(4, dy), textcoords="offset points",
                   color=COL[b], fontsize=9, va="center")
    a.set_xlabel("time, ns")
    a.set_ylabel(label)
    a.set_facecolor("white")
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
ax[0].axhline(ratio["buried"]["native"] * HOLD_FRACTION, ls="--", lw=0.8,
              color="#868E96")
ax[0].annotate("declared hold threshold, %.2f of native" % HOLD_FRACTION,
               xy=(0.02, ratio["buried"]["native"] * HOLD_FRACTION),
               xytext=(0, 4), textcoords="offset points", fontsize=8, color="#868E96")
fig.suptitle("Reverse-QTY series C kill criterion: %s, %.0f ns each"
             % (out["verdict"], ns["native"]), y=1.02, fontsize=11)
fig.tight_layout()
for ext in ("png", "svg"):
    p = os.path.join(FIGURES, "kill_criterion.%s" % ext)
    fig.savefig(p, dpi=400, facecolor="white", bbox_inches="tight")
    print("wrote", p)
plt.close(fig)


## Done

`md/results/kill_criterion.json` carries the verdict, the ratios it rests on, the
thresholds it was judged against, whether the look was one-sided, and the
per-position distances, so the decision can be re-read without rerunning anything.
The figure is in `figures/`.
